In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "sapientinc/HRM-Text-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).cuda().eval()

# synth,cot composite — reasoning / CoT style (see Disclaimer for other modes)
condition = "<|quad_end|><|object_ref_end|>"
prompt = f"<|im_start|>{condition}Explain why the sky is blue.<|im_end|>"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# Mark the prompt as a single bidirectional prefix block — see "PrefixLM mask" below.
inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=False))

/home/m.ortega/repositories/graph-representation-learning-for-llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 10016.84it/s]


<|im_start|><|quad_end|><|object_ref_end|>Explain why the sky is blue.<|im_end|>The sky appears blue due to the scattering of sunlight by air molecules. Shorter wavelengths of light (blue) scatter more easily than longer wavelengths (red), which is why the sky appears blue. This phenomenon is known as Rayleigh scattering. $\boxed{\text{The sky appears blue because shorter wavelengths of light (blue) scatter more easily than longer wavelengths (red) due to air molecules.}}$<|box_end|>
